In [1]:
!pip install -U dspy-ai litellm
# litellm is already used under the hood by dspy for OpenRouter, but upgrading helps

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 312.4/312.4 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.6/41.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 32.6 MB/s eta 0:00:00


In [2]:

import dspy
import getpass
import os

# Securely ask for your OpenRouter API key
OPENROUTER_API_KEY = getpass.getpass("Enter your OpenRouter API key: ")

# Set as env var (LiteLLM / DSPy will pick it up)
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

# Now create the LM using the unified dspy.LM
# Prefix with "openai/" because OpenRouter provides an OpenAI-compatible endpoint
lm = dspy.LM(
    "openai/z-ai/glm-4.5-air:free",          # ← note the "openai/" prefix here
    api_base="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,                # or rely on the env var
    max_tokens=2048,
    temperature=0.7
)

# Optional: enable caching if you run the same prompts multiple times
# lm = dspy.LMCache(lm)

# Set as default LM for DSPy
dspy.configure(lm=lm)

print("DSPy is now configured to use GLM-4.5-Air via OpenRouter! ✓")

Enter your OpenRouter API key: ··········
DSPy is now configured to use GLM-4.5-Air via OpenRouter! ✓


In [3]:

# Simple test signature
class BasicTest(dspy.Signature):
    """Answer concisely."""
    question: str = dspy.InputField()
    answer: str = dspy.OutputField(desc="Short answer")

# Use Predict module
generate = dspy.Predict(BasicTest)

result = generate(question="What is the capital of Japan?")
print(result.answer)

Tokyo


In [6]:

# ────────────────────────────────────────────────────────────────
# Interactive Scene → Image Prompt Generator
# ────────────────────────────────────────────────────────────────

class SceneToImagePrompt(dspy.Signature):
    """
    You are an expert image prompt engineer.You are an expert at turning loose, casual, or explicit user directions
    into ultra-detailed, vivid, high-quality prompts for modern text-to-image models
    (Flux, SD3, SDXL, Pony, etc.).

    ALWAYS follow these guidelines:
    - Start with strong cinematic composition (e.g. low-angle shot, dramatic perspective)
    - Emphasize body language, pose details, clothing (especially sheer/translucent fabrics)
    - Build seductive/mood-rich atmosphere with lighting, shadows, skin texture
    - Keep it anatomically realistic + professional erotic photography style
    - Tasteful yet explicit when appropriate — avoid vulgar/cartoon unless user says so
    - Output format: comma-separated descriptive tags (80–150 words), ready to copy-paste

    Examples:

    taste="photorealistic",
                user_input="a cat sitting on a windowsill",
                enhanced_prompt="You are an expert image prompt engineer. Transform this image concept into a photorealistic AI image generation prompt.

    **Input**:
    - Image Concept: "a cat sitting on a windowsill"
    - Quality Style: "photorealistic"

    **Output**:
    Ultra realistic photograph of a ginger cat sitting on a sunlit windowsill, detailed fur texture with individual hairs visible, sharp focus on the subject's eyes, natural lighting with soft shadows, 85mm lens with shallow depth of field, visible dust particles in the light beam, detailed wood grain on the windowsill, warm color temperature"


taste="photorealistic",
                user_input="a mountain landscape at sunset",
                enhanced_prompt="You are an expert image prompt engineer. Transform this image concept into a photorealistic AI image generation prompt.

**Input**:
- Image Concept: "a mountain landscape at sunset"
- Quality Style: "photorealistic"

**Output**:
Breathtaking photorealistic landscape of snow-capped mountains at golden hour, warm sunset colors reflecting on a serene lake, volumetric lighting with sun rays breaking through clouds, ultra high detail with visible rock textures and snow crystals, 8k resolution, deep depth of field with foreground elements to establish scale, atmospheric perspective on distant peaks"

"""
    user_directions: str = dspy.InputField(desc="Free-form directions from the user (e.g. 'she spreads legs wide, wears translucent, low camera angle')")

    detailed_prompt: str = dspy.OutputField(desc="Full, optimized image generation prompt")


# Create the generator
# (use ChainOfThought if you want to see reasoning steps)
generate = dspy.Predict(SceneToImagePrompt)
# generate = dspy.ChainOfThought(SceneToImagePrompt)   # ← try this for richer results

print("Ready! Describe the scene, pose, clothing, mood, angle, lighting etc.\n(type 'exit' or 'quit' to stop)\n")

while True:
    user_input = input("\nYour directions: ").strip()

    if user_input.lower() in ['exit', 'quit', 'qs', '']:
        print("Goodbye!")
        break

    try:
        result = generate(user_directions=user_input)
        print("\nGenerated prompt:\n")
        print(result.detailed_prompt)
        print("\n" + "-"*70 + "\n")

    except Exception as e:
        print(f"Error: {e}\nTry again or check your LM configuration.")

Ready! Describe the scene, pose, clothing, mood, angle, lighting etc.
(type 'exit' or 'quit' to stop)


Your directions: Low-angle shot of a woman with legs raised high in the air, revealing her figure through a sheer translucent dress, minimal white lace panties and bra visible, dramatic bedroom lighting with soft shadows highlighting her body contours, professional erotic photography style, realistic skin texture with visible pores, minimalist bed setting with simple white sheets, shallow depth of field focusing on the pelvic area, seductive pose emphasizing vulnerability and confidence, warm ambient lighting with contrast, detailed fabric rendering showing the delicate transparency of the dress, tasteful yet explicit composition maintaining artistic merit, high resolution image with realistic anatomy

Generated prompt:

Ultra-low angle shot of a woman with legs raised high in the air, sheer translucent ivory dress clinging to her curves with delicate lace trim, revealing minimal whi

KeyboardInterrupt: Interrupted by user